In [2]:
import httpx
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime
import time
import json
import psycopg
import os
import sys
import numpy as np
from dotenv import load_dotenv
from itertools import zip_longest

sys.path.append(os.path.abspath('./src'))
import db_functions as dbf
load_dotenv()
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST", "localhost")
port = os.getenv("DB_PORT", "5432")
dbname = os.getenv("DB_NAME")
conn_str = f"postgresql://{user}:{password}@{host}:{port}/{dbname}"
api_key = os.getenv("API_KEY")
api_url = 'https://api.stratz.com/graphql'
headers = {
    'User-Agent': 'STRATZ_API',
    "Authorization": f"Bearer {api_key}"
}

In [8]:
patches_response = httpx.get(f'{api_url}/constants/patch').json()

In [13]:
patches_response

[{'name': '6.70', 'date': '2010-12-24T00:00:00Z', 'id': 0},
 {'name': '6.71', 'date': '2011-01-21T00:00:00Z', 'id': 1},
 {'name': '6.72', 'date': '2011-04-27T00:00:00Z', 'id': 2},
 {'name': '6.73', 'date': '2011-12-24T00:00:00Z', 'id': 3},
 {'name': '6.74', 'date': '2012-03-10T00:00:00Z', 'id': 4},
 {'name': '6.75', 'date': '2012-09-30T00:00:00Z', 'id': 5},
 {'name': '6.76', 'date': '2012-10-21T00:00:00Z', 'id': 6},
 {'name': '6.77', 'date': '2012-12-15T00:00:00Z', 'id': 7},
 {'name': '6.78', 'date': '2013-05-30T00:00:00Z', 'id': 8},
 {'name': '6.79', 'date': '2013-11-24T00:00:00Z', 'id': 9},
 {'name': '6.80', 'date': '2014-01-27T00:00:00Z', 'id': 10},
 {'name': '6.81', 'date': '2014-04-29T00:00:00Z', 'id': 11},
 {'name': '6.82', 'date': '2014-09-24T00:00:00Z', 'id': 12},
 {'name': '6.83', 'date': '2014-12-17T00:00:00Z', 'id': 13},
 {'name': '6.84', 'date': '2015-04-30T21:00:00Z', 'id': 14},
 {'name': '6.85', 'date': '2015-09-24T20:00:00Z', 'id': 15},
 {'name': '6.86', 'date': '2015-12

In [10]:
patches_df = pd.DataFrame(patches_response)

In [16]:
patches_df['date'] = pd.to_datetime(patches_df['date'], format='ISO8601')

In [18]:
patches_df.dtypes

name                 object
date    datetime64[ns, UTC]
id                    int64
dtype: object

In [19]:
dbf.create_table_from_df(patches_df, 'patches', conn_str)

Table 'patches' created successfully.


In [20]:
dbf.insert_df_into_table(patches_df, 'patches', conn_str)

Data inserted into table 'patches' successfully.


In [2]:
heroes_response = httpx.get(f'{api_url}/constants/heroes').json()

In [29]:
heroes_df = pd.DataFrame(heroes_response[key] for key in heroes_response.keys())

In [30]:
heroes_df = heroes_df.drop('img', axis=1)
heroes_df = heroes_df.drop('icon', axis=1)
heroes_df = heroes_df.drop('legs', axis=1)

In [33]:
dbf.create_table_from_df(heroes_df, 'heroes', conn_str)

Table 'heroes' created successfully.


In [34]:
dbf.insert_df_into_table(heroes_df, 'heroes', conn_str)

Data inserted into table 'heroes' successfully.


In [ ]:
# Get game versions
query = """
    query {
        constants {
            gameVersions {
                id
                name
                asOfDateTime
            }
        }
    }
"""
result = dbf.query_stratz(query, headers=headers, api_url=api_url)
df = pd.DataFrame(result['data']['constants']['gameVersions'])
df['asOfDateTime'] = df['asOfDateTime'].apply(datetime.fromtimestamp)
dbf.create_table_from_df(df, 'patches', conn_str)
dbf.insert_df_into_table(df, 'patches', conn_str)

In [ ]:
## Get npc data from stratz
query = """
    query($gameVersionId: Short!) {
        constants {
            npcs(gameVersionId: $gameVersionId) {
                id
                name
                stat {
                    statusHealth
                    statusHealthRegen
                    attackDamageMin
                    attackDamageMax
                    attackRate
                    attackRange
                    movementSpeed
                    isNeutralUnitType
                    isAncient
                    teamName
                }
            }
        }
    }
"""
variables = {'gameVersionId': 182} #TODO: replace hardcoded value
result = dbf.query_stratz(query, headers=headers, api_url=api_url, variables=variables)
df = pd.DataFrame(result['data']['constants']['npcs'])
stats_df = pd.json_normalize(df['stat'])
df = df.join(stats_df).drop(columns=['stat'])
discard_patterns = [
    'thinker', 'companion', 'visual', 'sound', 'event', 
    'shmup', 'banana', 'target_dummy', 'looping', 'promo'
]
discard_regex = '|'.join(discard_patterns)
df_filtered = df[~df['name'].str.contains(discard_regex, case=False, na=False)]
df_filtered.dtypes
df_filtered.convert_dtypes(convert_integer=False).dtypes
dbf.create_table_from_df(df_filtered, 'npcs', conn_str, False)
dbf.insert_df_into_table(df_filtered, 'npcs', conn_str)

In [3]:
query = """
    query($gameVersionId: Short!) {
        constants {
            items(gameVersionId: $gameVersionId) {
                id
                        shortName
                displayName
                isSupportFullItem
                attributes {
                    name
                    value
                }
                stat {
                    cost
                    isRecipe
                    isSupport
                    behavior
                    manaCost
                    shopTags
                    needsComponents
                    itemResult
                    quality
                }
                components {
                    componentId
                }
            }
        }
    }
"""
variables = {'gameVersionId': 182} #TODO: replace hardcoded value
result = dbf.query_stratz(query, headers=headers, api_url=api_url, variables=variables)

In [6]:
result_json = result['data']['constants']['items']

In [8]:
df = pd.json_normalize(result_json)

In [ ]:
#TODO: 3 tables: item basic details, item attributes, item stats IF NEEDED think this through
#TODO: or extract the most important features from attributes and stats, and store rest in a table
df

,id,shortName,displayName,isSupportFullItem,attributes,components,stat.cost,stat.isRecipe,stat.isSupport,stat.behavior,stat.manaCost,stat.shopTags,stat.needsComponents,stat.itemResult,stat.quality,stat
0,1,blink,Blink Dagger,None,"[{'name': 'blink_damage_cooldown', 'value': '3...",None,2250.0,False,False,1.374395e+11,[0],teleport;mobility;escape,False,NaN,component,NaN
1,2,blades_of_attack,Blades of Attack,None,"[{'name': 'bonus_damage', 'value': '9'}]",None,450.0,False,False,2.000000e+00,None,damage;tutorial,False,NaN,component,NaN
2,3,broadsword,Broadsword,None,"[{'name': 'bonus_damage', 'value': '15'}]",None,1000.0,False,False,2.000000e+00,None,damage,False,NaN,component,NaN
3,4,chainmail,Chainmail,None,"[{'name': 'bonus_armor', 'value': '4'}]",None,550.0,False,False,2.000000e+00,None,armor,False,NaN,component,NaN
4,5,claymore,Claymore,None,"[{'name': 'bonus_damage', 'value': '20'}]",None,1350.0,False,False,2.000000e+00,None,damage,False,NaN,component,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
570,4207,recipe_great_famango,,None,None,"[{'componentId': 4204}, {'componentId': 4204},...",0.0,True,False,0.000000e+00,None,,False,4205.0,None,NaN
571,4208,recipe_greater_famango,,None,None,"[{'componentId': 4205}, {'componentId': 4205}]",0.0,True,False,0.000000e+00,None,,False,4206.0,None,NaN
572,4300,ofrenda,Beloved Memory,None,"[{'name': 'speed', 'value': '1000'}]",None,0.0,False,False,7.200000e+01,[0],None,False,NaN,None,NaN
573,4301,ofrenda_shovel,Scrying Shovel,None,None,None,0.0,False,False,1.342179e+08,[0],None,False,NaN,None,NaN
